# `eval` — the orchestrator, on Qdrant, over the 600 cases

Runs the code path that ships: `HealthcarePricingPipeline` — entity extraction →
Qdrant retrieval → code decision → validation → MRF price lookup → answer — with
MedGemma through **vLLM in this process**, then scores what it produced.

Every case gets a row: the seven key columns copied through untouched, and the run's
output beside them in new columns.

| generated column | what scores it |
|---|---|
| `generate_cpt`, `generate_hcpcs` | Sorensen-Dice, Jaccard vs the key; pivot hit |
| `generate_cpt_candidates`, `generate_hcpcs_candidates` | what retrieval offered (Recall@k, MRR) |
| `generate_row_ids` | Dice / Jaccard over MRF row ids, SQL row-set exactness, by stratum |
| `generate_rate`, `generate_cash`, `generate_matches` | rate citation fidelity (½¢ after rounding), right-number-wrong-row |
| `generate_hospital`, `generate_insurer`, `generate_medical`, `generate_medication` | precision / recall / F1 per field, then macro |
| `generate_intent` | accuracy and per-class F1 |
| `generate_answer` | abstention: correct silence vs false price |
| `generate_seconds`, `generate_error`, `generate_calls` | latency, failures, the raw text of every model call |

Cost: 600 cases × about five model calls. Run the ten-case smoke first — its
`generate_seconds` × 600 is the estimate.

If `pip` upgrades torch while installing vLLM, do **Runtime → Restart session**, then
re-run from §1. vLLM has to be imported against the torch it was built for.

## 1. Get the branch

Everything below imports from `orchestrator-eval`, cut from `no-router`.

In [ ]:
%cd /content
import os
REPO = "/content/HPT_Prototype"
if not os.path.isdir(REPO):
    !git clone -q https://github.com/adi274903/HPT_Prototype $REPO
%cd $REPO
!git fetch -q origin orchestrator-eval
!git checkout -q -B orchestrator-eval FETCH_HEAD
!git log --oneline -1
!ls pt_healthcare/eval_pipeline.py eval/pt_eval_v2.jsonl eval/pt_eval_v2_targets.jsonl

## 2. Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q vllm qdrant-client sentence-transformers duckdb pandas huggingface_hub

import qdrant_client, sentence_transformers, vllm
print("vllm", vllm.__version__, "| qdrant-client", qdrant_client.__version__)

## 3. Drive, the Qdrant store, the MRF

The store is the snapshot on Drive, extracted once to `/content/New_PT_DB`. The
collections it holds are the ones the app queries — `cpt_medte` and `hcpcs_medte` —
so there is no catalog CSV to stage and nothing to embed.

In [ ]:
import os
import colab_run

colab_run.mount_drive()

QDRANT_PATH = "/content/New_PT_DB"
if not os.path.isdir(QDRANT_PATH):
    QDRANT_PATH = colab_run.stage_db()          # copy + extract, ~2 GB
else:
    print("already staged:", QDRANT_PATH)

MRF_CSV = colab_run.resolve_mrf_csv()           # cleaned 2.4 GB csv on Drive
print("qdrant:", QDRANT_PATH)
print("mrf   :", MRF_CSV, round(os.path.getsize(MRF_CSV) / 1e9, 2), "GB")

# the pipeline reads these from the environment; export what we resolved
os.environ["PT_QDRANT_PATH"] = QDRANT_PATH
os.environ["PT_MRF_CSV"] = MRF_CSV

## 4. The generator — vLLM, in this process

`load_models(backend="vllm")` returns `(generator, embedder)`. The embedder is the
MedTE sentence-transformer the Qdrant collections were built with; it is not
optional, retrieval has no other path.

In [ ]:
from pt_healthcare import config
from pt_healthcare.data import login_huggingface
from pt_healthcare.models import load_models

login_huggingface()                    # gated MedGemma / MedTE weights
print("generator:", config.medgemma_model_id())
print("embedder :", config.embedding_model_id())

pipe, embed_model = load_models(device="cuda", backend="vllm")
print("loaded:", type(pipe).__name__, "|", type(embed_model).__name__)

## 5. Retriever, MRF frame, pipeline

`RecordingEngine` wraps `MedGemmaEngine` and keeps the raw text of every call — that
is the generation data: not the parsed JSON, but what the model emitted, per step,
with its own latency.

In [ ]:
from pt_healthcare.data import get_qdrant_client, load_mrf_data
from pt_healthcare.eval_pipeline import GENERATED_COLUMNS, RecordingEngine, generate, score
from pt_healthcare.llm import MedGemmaEngine
from pt_healthcare.pipeline import HealthcarePricingPipeline
from pt_healthcare.retrieval import CodeRetriever

client = get_qdrant_client(QDRANT_PATH)
mrf_data = load_mrf_data(MRF_CSV)
TOP_K = config.top_k()

engine = RecordingEngine(MedGemmaEngine(pipe))
pipeline = HealthcarePricingPipeline(
    retriever=CodeRetriever(embed_model, client, top_k=TOP_K),
    engine=engine,
    mrf_data=mrf_data,
    top_k=TOP_K,
    printer=lambda _message: None,        # 600 cases, seven steps each: no printing
    display_fn=lambda *_args, **_kwargs: None,
)

print("collections:", [c.name for c in client.get_collections().collections])
print("MRF rows   :", len(mrf_data), "| top_k:", TOP_K)

## 6. One query, by hand

Cheaper than a smoke run, and it shows the shape of what the metrics read.

In [ ]:
probe = "What does a colonoscopy cost at UPMC Presbyterian with Highmark BCBS?"
result = pipeline.run(probe, verbose=False)

print("cpt   :", result["output"]["cpt_list"])
print("hcpcs :", result["output"]["hcpcs_list"])
print("rows  :", result["code_plausible"].get("match_count"))
print("answer:", (result.get("answer") or "")[:300].replace("\n", " "))
for call in engine.calls:
    print(f"  {call['step']:12} {call['ms']:7.0f} ms  {len(call['text']):5} chars")

## 7. Smoke — ten cases, one per subtype

Three things to check before the full run: `generate_cpt` is not empty, row ids come
back as ids, and `generate_seconds` looks like real inference and not a stub.

In [ ]:
rows = generate(pipeline=pipeline, dataset="eval",
                out="/content/eval_smoke.jsonl", per_subtype=1, fresh=True)

first = rows[0]
for column in GENERATED_COLUMNS:
    print(f"{column:28} {str(first[column])[:86]}")

## 8. The full 600

Every finished case is appended to `/content/eval_generations.jsonl` and skipped on
the next run, so a dropped runtime costs the case in flight — re-run and it
continues. `fresh=True` starts over.

`with_intent=True` adds one routing call per case for the intent column. This branch
has no router stage; without the flag the report says *not measured* rather than
printing a zero.

In [ ]:
rows = generate(pipeline=pipeline, dataset="eval",
                out="/content/eval_generations.jsonl",
                csv_out="/content/eval_generations.csv",
                with_intent=True)
print(len(rows), "cases on disk")

## 9. Score the columns

No model and no Qdrant here — it judges what is already on disk, so changing a
metric costs seconds instead of another 600-case run.

In [ ]:
report = score(rows="/content/eval_generations.jsonl", dataset="eval",
               mrf_data=mrf_data,                  # prices the key off the frame in memory
               report_path="/content/eval_report.json",
               scores_out="/content/eval_scores.jsonl")

from pt_healthcare.eval_runner import format_report
print(format_report(report))

## 10. Where the numbers came from

Per subtype first: an aggregate hides the one that collapsed.

In [ ]:
import pandas as pd

generations = pd.read_csv("/content/eval_generations.csv")
scores = pd.read_json("/content/eval_scores.jsonl", lines=True)
scores["dice"] = scores["codes"].apply(lambda value: value.get("dice_mrf"))
scores["rows_dice"] = scores["rows"].apply(lambda value: value.get("dice_raw"))
scores["pivot_hit"] = scores["codes"].apply(lambda value: value.get("pivot_hit"))

print(scores.groupby("subtype")[["dice", "rows_dice", "pivot_hit"]].mean().round(3))
print()
print(scores.nsmallest(10, "dice")[["eval_id", "subtype", "dice", "rows_dice"]].to_string(index=False))
generations[["eval_id", "generate_cpt", "generate_row_ids",
             "generate_rate", "generate_seconds"]].head(10)

## 11. Keep the run

Colab forgets `/content`; Drive does not.

In [ ]:
import shutil, time

DEST = "/content/drive/MyDrive/Price_Transparency/eval_runs"
os.makedirs(DEST, exist_ok=True)
stamp = time.strftime("%Y%m%d_%H%M")

for name in ("eval_generations.jsonl", "eval_generations.csv",
             "eval_report.json", "eval_scores.jsonl"):
    source = os.path.join("/content", name)
    if os.path.exists(source):
        shutil.copy(source, os.path.join(DEST, f"{stamp}_{name}"))

print("saved to", DEST)
print(sorted(os.listdir(DEST))[-4:])

## Notes

**Reading the headline.** `dice_mrf` is scored against the codes that have MRF rows,
so returning the right service among five plausible ones scores badly by construction
— the ceiling is structural, not a bug. Read `pivot_hit` with it. `sql_exact` is the
negotiated-rate check: the row set has to match, not just the figure.

**Resume, don't restart.** Runtime died at case 400? Re-run §8 — it picks up at 401.
Only `fresh=True` throws work away. For a slice, use a separate `out`:

```python
rows = generate(pipeline=pipeline, dataset="eval", subtypes=["C1", "D2"],
                per_subtype=20, out="/content/eval_slice.jsonl")
```

**CLI instead of the notebook.** Same two phases, from a fresh process — but it opens
its own Qdrant client, so free the lock first (§12).

```python
!python -m pt_healthcare.eval_pipeline generate --qdrant-path /content/New_PT_DB \
    --mrf "$PT_MRF_CSV" --out /content/eval_generations.jsonl --with-intent
!python -m pt_healthcare.eval_pipeline score --rows /content/eval_generations.jsonl \
    --db .cache/mrf_eval.duckdb --report /content/eval_report.json
```

## 12. Free the GPU and the store

Local-mode Qdrant takes a file lock on the store directory. Any second client — a re-run of §5, the CLI above — fails with *already accessed by another instance* until this cell has run.

In [ ]:
import gc

try:
    client.close()
except Exception as error:
    print("client:", error)

for obj in gc.get_objects():
    if type(obj).__name__ in ("QdrantClient", "QdrantLocal"):
        try:
            obj.close()
        except Exception:
            pass

del pipeline, engine, pipe, embed_model, client
gc.collect()

import torch
torch.cuda.empty_cache()
print("freed; torch holds", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")